Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [ ]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
# df_res = (
#     bd.load_meas_from_excel(
#         # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
#         "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
#         study_folder="CFR",
#         str_cols_to_arrays=["Airway resistance (%)"],
#         use_csv=True,
#         date_cols=["Day"],
#         bypass_sanity_checks=True,
#     )
#     .drop(columns=["Healthy FEV1 (L)"])
#     .rename(columns={"Day": "Date Recorded"})
# )

# Merging
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [2]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
# df = bd.load_meas_from_excel("AR_19_data_with_best_FEV1", study_folder="CFR", str_cols_to_arrays=["Airway resistance (%)"])
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(FEV1|pred_FEV1)",
        "P(HFEV1|FEF2575, bFEV1)",
    ],
)
print(f"Shape: {df.shape}")

Shape: (2037, 21)


In [4]:
# Process

# Keep only values from 2023
# df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]

df23 = df

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df23[AC.name] = df23[AR.name].apply(lambda arr: arr[::-1])
df23["ecFEV1 % Predicted (clipped)"] = df23["ecFEV1 % Predicted"].clip(upper=100)
df23["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df23, AC)
df["P(ppFEV1|AC) ratioed"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# Airway conductance

In [36]:
## FILL ##
ratioed = False
prctile = 10

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
# df_to_plot = df23[df23["P(ppFEV1|AC)"] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

ppfev1_row = "ecFEV1 % Predicted (clipped)"
# ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df_to_plot, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)}) clipped"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 169
# Moderate: 33
# Severe: 2
35.60473898878335


# FEV1 % personalised predicted (Non saturating)

In [13]:
ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
df["mean P(FEV1|pred_FEV1)"] = df.apply(
    lambda row: ecFEV1.get_mean(row["P(FEV1|pred_FEV1)"]), axis=1
)

df["FEV1%PersPred"] = df["FEV1"] / df["mean P(FEV1|pred_FEV1)"] * 100

In [14]:
import plotly.graph_objects as go

In [15]:
def plot_dumbell_for_df_model_ppfev1(fig, df, measures, col):
    ac_mean = measures[0]
    baseline = measures[1]

    mask = df["measure"] == ac_mean
    fig.add_trace(
        go.Scatter(
            x=df[mask]["value"],
            y=df[mask]["ID"],
            mode="markers",
            marker=dict(color="red", size=4),
            name="ecFEV1 % pers. pred."
        ),
        row=1,
        col=col,
    )

    mask = df["measure"] == baseline
    ecfev1_prct_pred = df[mask]["value"]
    # Where above 100, set to 100
    # ecfev1_prct_pred = np.clip(ecfev1_prct_pred, 0, 100)
    fig.add_trace(
        go.Scatter(
            x=ecfev1_prct_pred,
            y=df[mask]["ID"],
            mode="markers",
            name="ecFEV1 % predicted",
            marker=dict(size=4, color="blue"),
        ),
        row=1,
        col=col,
    )

In [16]:
def get_dumbell_plot_data_model_ppfev1(df, ac_row, ppfev1_row="ecFEV1 % Predicted"):
    # Avoid modifying the original dataframe
    df_res = df.copy()

    ids_sorted = df_res.sort_values("ppFEV1 - pppFEV1", ascending=False)["ID"].values

    df_melted = (
        df_res.melt(
            id_vars=["ID"],
            value_vars=[ppfev1_row, ac_row],
            var_name="measure",
            value_name="value",
        )
        .set_index("ID")
        .loc[ids_sorted]
        .reset_index()
    )

    return df_melted, df_res, ids_sorted

In [ ]:
## FILL ##
prctile = 90
prctile = 0

title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, FEV1%PersPred {prctile:.0f}th prctile"
ac_col = "FEV1%PersPred"
ppfev1_row = "ecFEV1 % Predicted"

df["ppFEV1 - pppFEV1"] = df["FEV1 % Predicted"] - df["FEV1%PersPred"]
col = "ppFEV1 - pppFEV1"
t = df[col].abs().quantile(prctile / 100)
df_to_plot = df[df[col].abs() > t]


df_to_plot, _, _ = get_dumbell_plot_data_model_ppfev1(
    df_to_plot, ac_col, ppfev1_row
)

mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

# Plot the three groups
fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)
title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"
plot_dumbell_for_df_model_ppfev1(fig, df_mild, [ac_col, ppfev1_row], 3)
plot_dumbell_for_df_model_ppfev1(fig, df_moderate, [ac_col, ppfev1_row], 2)
plot_dumbell_for_df_model_ppfev1(fig, df_severe, [ac_col, ppfev1_row], 1)

fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

13.3130788065074


In [20]:
df[df['FEV1'] > df['best FEV1']]

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,idx FEV1,idx FEF2575%FEV1,idx best FEV1,"P(HFEV1|FEF2575, bFEV1)",P(FEV1|pred_FEV1),P(FEV1_obs|pred_FEV1),Airway resistance (%),mean P(FEV1|pred_FEV1),FEV1%PersPred,ppFEV1 - pppFEV1
18,B155953,52,162,1.25,0.34,1.03,Male,2019-01-01,1.25,0.34,...,25,13,20,"[1.80455334e-06, 4.53977566e-06, 7.61894763e-0...","[8.339495e-199, 1.14780163e-175, 2.23757183e-1...",2.763200e-05,"[3.84619494e-08, 2.20681798e-07, 1.1182584e-06...",3.037736,41.149062,-1.425138
125,B156571,39,153,2.14,1.14,2.13,Female,2019-01-01,2.14,1.14,...,42,26,42,"[3.73482304e-152, 1.19776524e-138, 8.65276647e...","[0.0, 2.376e-321, 4.63106747e-300, 1.54209242e...",1.541371e-02,"[0.00109619034, 0.00337333325, 0.00860724282, ...",2.644187,80.932239,-0.906821
149,B156823,34,156,2.14,0.99,2.00,Female,2019-01-01,2.14,0.99,...,42,23,40,"[6.86411977e-130, 2.27845864e-117, 1.64644753e...","[3.16e-322, 4.36598241e-299, 8.51123361e-278, ...",8.532859e-03,"[0.000269920973, 0.000837122183, 0.00236114527...",2.815059,76.019718,-1.817890
151,B156825,29,159,3.16,2.68,3.00,Female,2019-01-01,3.16,2.68,...,63,42,60,"[0.0, 0.0, 0.0, 0.0, 0.0, 3.14582243e-311, 1.6...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",9.548535e-02,"[0.156248541, 0.15619409, 0.148362261, 0.13552...",3.280266,96.333661,6.299398
319,B157811,42,184,1.88,0.71,1.84,Male,2019-01-01,1.88,0.71,...,37,18,36,"[8.02571408e-90, 1.52723984e-79, 5.92592063e-7...","[3.70897335e-282, 5.10482429e-259, 9.95155503e...",6.582085e-06,"[2.84172021e-08, 1.29015213e-07, 5.04463805e-0...",4.353222,43.186404,-1.362648
325,B157849,38,171,1.13,0.37,1.09,Male,2019-01-01,1.13,0.37,...,22,16,21,"[8.45760411e-09, 4.36702529e-08, 1.10160742e-0...","[3.90856539e-201, 5.37953165e-178, 1.04870808e...",1.334135e-07,"[2.77549898e-10, 1.34105283e-09, 5.70707603e-0...",3.806987,29.682264,-0.898029
330,B157861,50,170,2.84,2.30,2.77,Female,2019-01-01,2.84,2.30,...,56,40,55,"[0.0, 2.73937421e-310, 3.85481679e-292, 1.5161...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",5.777679e-02,"[0.050370543, 0.0678784617, 0.0815494571, 0.09...",3.149915,90.161159,3.207911
338,B157877,58,181,4.64,4.75,4.46,Male,2019-01-01,4.64,4.75,...,92,51,89,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",9.704305e-02,"[0.361477454, 0.248513834, 0.158590486, 0.1026...",4.652452,99.732345,22.048578
339,B157878,33,158,2.44,1.10,2.39,Female,2019-01-01,2.44,1.10,...,48,22,47,"[6.69945002e-216, 3.66159421e-200, 4.93082976e...","[0.0, 0.0, 0.0, 0.0, 0.0, 7.943763e-309, 4.730...",2.146302e-02,"[0.000954368517, 0.00320208158, 0.00916951762,...",2.946215,82.818128,-0.938159
345,B157887,32,173,3.56,1.76,3.30,Male,2019-01-01,3.56,1.76,...,71,24,66,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.647456e-02,"[0.0033320666, 0.00858511079, 0.0204137086, 0....",4.128319,86.233642,-0.968892
